> **Note:** If you have opened this notebook before, Colab may show a cached version. To get the latest, go to **File → Open notebook → GitHub**, search for `anwai98/prompt-to-plot`, and reopen it from there.

# LLM-Assisted Data Plotting with Gemini

This notebook teaches you how to use a Large Language Model (Gemini) to **generate plotting code from plain English descriptions**.

Instead of looking up matplotlib/seaborn/plotly syntax, you describe what you want and the LLM writes the code for you.

### What this notebook does
You describe a plot in plain English → Gemini writes the Python code → the notebook runs it and shows the result. If the code fails, it automatically sends the error back to Gemini and tries again.

### What you'll learn
- How to connect to Gemini via the Google Generative AI SDK
- How to pass data context to an LLM so it understands your dataset
- How to generate and run plotting code for **matplotlib**, **seaborn**, and **plotly**
- How to write better prompts for more precise plots

### Prerequisites
- A free Gemini API key from [Google AI Studio](https://aistudio.google.com/app/apikey)
- Store your key in Colab Secrets (the key icon in the left sidebar) under the name `GEMINI_API_KEY`

## Contents

1. [Install & Import Dependencies](#Install-&-Import-Dependencies)
2. [Dummy Datasets](#Dummy-Datasets)
3. [Helper Functions](#Helper-Functions)
4. [Matplotlib Examples](#Matplotlib-Examples)
5. [Seaborn Examples](#Seaborn-Examples)
6. [Plotly Examples](#Plotly-Examples)
7. [Biological Data: Gene Expression Examples](#Biological-Data:-Gene-Expression-Examples)
8. [Tips for Writing Better Prompts](#Tips-for-Writing-Better-Prompts)
9. [Troubleshooting](#Troubleshooting)
10. [Your Turn](#Your-Turn)


## Install & Import Dependencies

In [3]:
!pip install -q -U google-genai

In [ ]:
import re
import textwrap
import time

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from google import genai
from google.genai import errors as genai_errors
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=GEMINI_API_KEY)
MODEL = 'gemini-flash-lite-latest'

print('Setup complete. Gemini is ready.')

## Dummy Datasets

We create five datasets inline. In a real project you would load a CSV with `pd.read_csv()`.

| Dataset | Description | Good for | Created in |
|---|---|---|---|
| `df_sales` | Monthly sales by product category | bar charts, line charts | this section |
| `df_students` | Exam scores, study hours, grade group | histograms, box plots, heatmaps | this section |
| `df_countries` | GDP, population, life expectancy | scatter plots, bubble charts | this section |
| `df_volcano` | 200 genes with fold change and p-value | volcano plots | Gene Expression section |
| `df_heatmap` | Top 20 genes x 6 samples | expression heatmaps | Gene Expression section |

In [ ]:
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
rng = np.random.default_rng(42)

df_sales = pd.DataFrame({
    'Month': months,
    'Electronics': rng.integers(15000, 35000, 12),
    'Clothing': rng.integers(8000, 20000, 12),
    'Groceries': rng.integers(20000, 40000, 12),
    'Furniture': rng.integers(5000, 15000, 12),
})
df_sales['Total'] = df_sales[['Electronics', 'Clothing', 'Groceries', 'Furniture']].sum(axis=1)

print('df_sales shape:', df_sales.shape)
df_sales.head()

In [ ]:
n = 120
df_students = pd.DataFrame({
    'student_id': range(1, n + 1),
    'exam_score': np.clip(rng.normal(65, 15, n), 0, 100).round(1),
    'study_hours': np.clip(rng.normal(6, 2.5, n), 0, 15).round(1),
    'assignments': np.clip(rng.normal(75, 12, n), 0, 100).round(1),
    'attendance': np.clip(rng.normal(80, 10, n), 0, 100).round(1),
    'grade_group': rng.choice(['A', 'B', 'C', 'D'], n, p=[0.2, 0.35, 0.3, 0.15]),
})

print('df_students shape:', df_students.shape)
df_students.head()

In [ ]:
country_names = [f'Country_{i}' for i in range(1, 51)]
continents = rng.choice(
    ['Africa', 'Asia', 'Europe', 'Americas', 'Oceania'],
    50,
    p=[0.25, 0.3, 0.25, 0.15, 0.05]
)

df_countries = pd.DataFrame({
    'country': country_names,
    'continent': continents,
    'gdp_per_capita': np.clip(rng.lognormal(9, 1.2, 50), 500, 80000).round(),
    'life_expectancy': np.clip(rng.normal(70, 10, 50), 45, 85).round(1),
    'population_M': np.clip(rng.lognormal(3, 1.5, 50), 0.5, 1400).round(1),
    'co2_per_capita': np.clip(rng.lognormal(1.5, 0.8, 50), 0.1, 20).round(2),
})

print('df_countries shape:', df_countries.shape)
df_countries.head()

## Helper Functions

Two functions power the notebook:

1. **`build_prompt`** — Describes your DataFrame to Gemini (column names, types, sample rows) plus your request, and instructs the model to return only runnable Python code.
2. **`ask_gemini_to_plot`** — Calls the Gemini API, extracts the code block from the response, and executes it.

In [ ]:
def build_prompt(df, df_name, user_request, library_hint=''):
    col_info = ', '.join(
        [f'{col} ({dtype})' for col, dtype in zip(df.columns, df.dtypes)]
    )
    sample_rows = df.head(3).to_string(index=False)
    library_line = f'Use the {library_hint} library.' if library_hint else ''

    return textwrap.dedent(f"""
        You are a Python data visualisation expert.
        A pandas DataFrame called `{df_name}` is already defined with these columns:
          {col_info}

        First 3 rows:
        {sample_rows}

        Task: {user_request}
        {library_line}

        Rules:
        - Return ONLY a Python code block (```python ... ```).
        - Do NOT include any explanation or text outside the code block.
        - The DataFrame `{df_name}` is already loaded — do not reload or redefine it.
        - All required libraries (matplotlib, seaborn, plotly, pandas, numpy) are already imported.
        - End matplotlib/seaborn plots with `plt.tight_layout(); plt.show()`.
        - End plotly plots with `fig.show()`.
    """).strip()


def build_fix_prompt(code, error):
    return textwrap.dedent(f"""
        The following Python plotting code raised an error when executed:

        ```python
        {code}
        ```

        Error:
        {error}

        Fix the code so it runs without errors. Return ONLY the corrected Python code block.
    """).strip()


def extract_code(text):
    match = re.search(r'```(?:python)?\n(.*?)```', text, re.DOTALL)
    return match.group(1).strip() if match else text.strip()


def parse_retry_wait(exc, default=60):
    match = re.search(r'retry in (\d+(?:\.\d+)?)s', str(exc))
    return int(float(match.group(1))) + 5 if match else default


def call_gemini(prompt):
    for attempt in range(5):
        try:
            response = client.models.generate_content(model=MODEL, contents=prompt)
            return response
        except genai_errors.ClientError as e:
            if '429' in str(e) or 'quota' in str(e).lower():
                wait = parse_retry_wait(e)
                print(f'Rate limit hit, waiting {wait}s before retry {attempt + 1}/5...')
                time.sleep(wait)
            else:
                raise
        except genai_errors.ServerError:
            print(f'Service unavailable, retrying {attempt + 1}/5...')
            time.sleep(5)
    return None


def ask_gemini_to_plot(
    df, df_name, user_request, library_hint='', print_code=True, print_prompt=False
):
    prompt = build_prompt(df, df_name, user_request, library_hint)

    if print_prompt:
        print('Prompt sent to Gemini:')
        print(prompt)
        print()

    response = call_gemini(prompt)

    if response is None:
        print('Failed to get a response after 5 attempts.')
        return

    code = extract_code(response.text)

    if print_code:
        print(code)

    exec_globals = {
        df_name: df,
        'pd': pd,
        'np': np,
        'plt': plt,
        'matplotlib': matplotlib,
        'sns': sns,
        'px': px,
        'go': go,
    }

    try:
        exec(code, exec_globals)  # noqa: S102
    except Exception as e:
        print(f'Execution error: {e}. Asking Gemini to fix...')
        plt.close('all')
        fix_response = call_gemini(build_fix_prompt(code, e))
        if fix_response is None:
            print('Failed to get a fix.')
            return
        fixed_code = extract_code(fix_response.text)
        if print_code:
            print('Fixed code:')
            print(fixed_code)
        exec(fixed_code, exec_globals)  # noqa: S102

### How it works

Every time you call `ask_gemini_to_plot`, three things happen:

1. **`build_prompt`** packages your DataFrame — column names, types, and the first 3 rows — together with your plain-English request into a structured prompt. This gives Gemini enough context to write correct code without seeing the full dataset.

2. **`call_gemini`** sends the prompt to the API and handles rate limits and server errors automatically, retrying up to 5 times with the wait time recommended by the API.

3. **`exec`** runs the returned code in a controlled namespace that includes your DataFrame and all plotting libraries. If the code raises an error, it is sent back to Gemini for an automatic fix.

To inspect the exact prompt being sent, pass `print_prompt=True` to any `ask_gemini_to_plot` call.

## Matplotlib Examples

Matplotlib is the most widely used Python plotting library — it gives you full control over every detail of a static figure.

**Example 1:** Ask Gemini to generate a grouped bar chart showing monthly sales broken down by product category.

In [ ]:
ask_gemini_to_plot(
    df=df_sales,
    df_name='df_sales',
    user_request=(
        'Create a grouped bar chart showing monthly sales for each product category '
        '(Electronics, Clothing, Groceries, Furniture). '
        'Use a distinct color per category. Add a legend, axis labels, and a title.'
    ),
    library_hint='matplotlib',
)

**Example 2:** Ask Gemini to plot total monthly sales as a line chart and annotate the peak month.

In [ ]:
ask_gemini_to_plot(
    df=df_sales,
    df_name='df_sales',
    user_request=(
        'Plot the Total monthly sales as a line chart. '
        'Mark the month with the highest total sales with a red dot and annotate it '
        'with the exact value. Add axis labels and a title.'
    ),
    library_hint='matplotlib',
)

## Seaborn Examples

Seaborn is built on top of matplotlib and is designed for **statistical visualisation** — it makes it easy to show distributions, relationships, and group comparisons.

**Example 3:** Ask Gemini to plot the distribution of exam scores as a histogram with a KDE overlay and a mean line.

In [ ]:
ask_gemini_to_plot(
    df=df_students,
    df_name='df_students',
    user_request=(
        'Plot a histogram of exam_score with a KDE (kernel density estimate) overlay. '
        'Use 20 bins and a soft blue color palette. Add a vertical dashed line at the mean score. '
        'Add axis labels and a title.'
    ),
    library_hint='seaborn',
)

**Example 4:** Ask Gemini to compare exam score distributions across grade groups using a box plot.

In [ ]:
ask_gemini_to_plot(
    df=df_students,
    df_name='df_students',
    user_request=(
        'Create a box plot showing the distribution of exam_score for each grade_group '
        '(A, B, C, D). Order the groups from A to D on the x-axis. '
        'Color each group differently. Add axis labels and a title.'
    ),
    library_hint='seaborn',
)

**Example 5:** Ask Gemini to compute and visualise the correlation matrix of all numeric columns as a heatmap.

In [ ]:
ask_gemini_to_plot(
    df=df_students,
    df_name='df_students',
    user_request=(
        'Compute the correlation matrix of all numeric columns and display it as a heatmap. '
        'Annotate each cell with the correlation value rounded to 2 decimal places. '
        'Use a diverging color palette centered at 0. Add a title.'
    ),
    library_hint='seaborn',
)

## Plotly Examples

Plotly produces **interactive** charts — you can hover for exact values, zoom in, pan, and toggle series on/off.

**Example 6:** Ask Gemini to create an interactive bubble chart of GDP vs life expectancy, with bubble size representing population.

In [ ]:
ask_gemini_to_plot(
    df=df_countries,
    df_name='df_countries',
    user_request=(
        'Create an interactive scatter plot with gdp_per_capita on the x-axis (log scale) '
        'and life_expectancy on the y-axis. '
        'Size each bubble by population_M and color by continent. '
        'Add hover labels that show the country name, GDP, and life expectancy. '
        'Add axis labels and a title.'
    ),
    library_hint='plotly',
)

**Example 7:** Ask Gemini to create an interactive stacked bar chart of monthly sales broken down by category.

In [ ]:
ask_gemini_to_plot(
    df=df_sales,
    df_name='df_sales',
    user_request=(
        'Create an interactive stacked bar chart showing monthly sales broken down by '
        'Electronics, Clothing, Groceries, and Furniture. '
        'Show the Month on the x-axis. Add a legend, axis labels, and a title.'
    ),
    library_hint='plotly',
)

## Biological Data: Gene Expression Examples

Gene expression data measures how active each gene is under different conditions. The two datasets below mimic what you'd get from an RNA-seq experiment comparing a treated sample against a control.

| Dataset | Description | Good for |
|---|---|---|
| `df_volcano` | 200 genes with fold change and p-value | volcano plots |
| `df_heatmap` | Top 20 genes x 6 samples (3 ctrl, 3 treated) | expression heatmaps |

In [ ]:
n_genes = 200
gene_names = [f'GENE_{i:03d}' for i in range(1, n_genes + 1)]
log2fc = rng.normal(0, 1.5, n_genes)
pval = rng.uniform(0, 1, n_genes)

sig_mask = (np.abs(log2fc) > 1.5) & (pval < 0.05)
pval[sig_mask] = rng.uniform(0, 0.001, sig_mask.sum())

df_volcano = pd.DataFrame({
    'gene': gene_names,
    'log2fc': log2fc.round(3),
    'pval': pval.round(5),
    'neg_log10_pval': (-np.log10(pval + 1e-10)).round(3),
    'significant': (np.abs(log2fc) > 1) & (pval < 0.05),
})

print('df_volcano shape:', df_volcano.shape)
df_volcano.head()

samples = ['ctrl_1', 'ctrl_2', 'ctrl_3', 'treat_1', 'treat_2', 'treat_3']
top_genes = (
    df_volcano.nlargest(10, 'neg_log10_pval')['gene'].tolist()
    + df_volcano.nsmallest(10, 'log2fc')['gene'].tolist()
)
expr_data = rng.normal(0, 1, (20, 6))
expr_data[:10, 3:] += 2
expr_data[10:, 3:] -= 2

df_heatmap = pd.DataFrame(expr_data.round(2), index=top_genes, columns=samples)

print('df_heatmap shape:', df_heatmap.shape)
df_heatmap.head()

**Example 8:** Ask Gemini to create a volcano plot — a standard plot in genomics that shows which genes are statistically significant and strongly up/down-regulated.

In [ ]:
ask_gemini_to_plot(
    df=df_volcano,
    df_name='df_volcano',
    user_request=(
        'Create a volcano plot with log2fc on the x-axis and neg_log10_pval on the y-axis. '
        'Color points by the significant column: red for significant, grey for not significant. '
        'Add dashed threshold lines at x = -1 and x = 1, and at y = -log10(0.05). '
        'Label the top 5 most significant genes by name. Add axis labels and a title.'
    ),
    library_hint='matplotlib',
)

**Example 9:** Ask Gemini to create a gene expression heatmap showing the top differentially expressed genes across control and treated samples.

In [ ]:
ask_gemini_to_plot(
    df=df_heatmap,
    df_name='df_heatmap',
    user_request=(
        'Create a clustered heatmap of the gene expression matrix. '
        'Rows are genes, columns are samples (ctrl_1-3 and treat_1-3). '
        'Use a red-blue diverging palette centered at 0. '
        'Add a title and make gene names readable on the y-axis.'
    ),
    library_hint='seaborn',
)

**Example 10:** Ask Gemini to create an interactive volcano plot so you can hover over individual genes to see their name, fold change, and p-value.

In [ ]:
ask_gemini_to_plot(
    df=df_volcano,
    df_name='df_volcano',
    user_request=(
        'Create an interactive volcano plot with log2fc on the x-axis and neg_log10_pval on the y-axis. '
        'Color points by the significant column: red for significant, grey for not significant. '
        'Add hover tooltips showing the gene name, log2fc, and pval. '
        'Add dashed threshold lines at x = -1, x = 1, and y = -log10(0.05). '
        'Add axis labels and a title.'
    ),
    library_hint='plotly',
)

## Tips for Writing Better Prompts

| Vague prompt | Better prompt |
|---|---|
| "plot the sales" | "Create a bar chart of total monthly sales with months on the x-axis" |
| "show the distribution" | "Show a histogram of exam_score with 20 bins and a KDE overlay" |
| "scatter plot" | "Scatter plot: x = gdp_per_capita (log scale), y = life_expectancy, colored by continent" |

Useful things to include: which columns to use for x/y/color/size, which library, axis scaling, annotations, color preferences, and any data transformations.

## Troubleshooting

| Problem | Fix |
|---|---|
| Plot looks like an old version | Colab is showing a cached notebook. Go to **File → Open notebook → GitHub**, search `anwai98/prompt-to-plot`, and reopen fresh. |
| `Rate limit hit` messages | Normal on the free tier. The notebook waits automatically — just leave it running. |
| `Failed to get a response after 5 attempts` | Daily quota may be exhausted. Wait a few minutes or try the next day. |
| `Execution error: ... Asking Gemini to fix` | The generated code had a bug — the notebook sends it back to Gemini automatically. If it fails again, re-run the cell. |
| `Secret not found` | Make sure the secret is named exactly `GEMINI_API_KEY` (case-sensitive) and **Notebook access** is toggled on. |

## Your Turn

Edit the three variables below and run the cell.

In [ ]:
MY_DATASET = df_students          # choose: df_sales, df_students, df_countries, df_volcano, df_heatmap
MY_DATASET_NAME = 'df_students'   # match the variable name above as a string
MY_LIBRARY = 'seaborn'            # choose: 'matplotlib', 'seaborn', 'plotly'
MY_REQUEST = """
Create a scatter plot of study_hours vs exam_score, colored by grade_group.
Add a linear regression trend line for each group.
Add axis labels and a title.
"""

ask_gemini_to_plot(
    df=MY_DATASET,
    df_name=MY_DATASET_NAME,
    user_request=MY_REQUEST,
    library_hint=MY_LIBRARY,
)

In [ ]:
# Run this first to see the exact column names and types for your chosen dataset
print("Columns:", MY_DATASET.columns.tolist())
print()
print(MY_DATASET.dtypes)